<a href="https://colab.research.google.com/github/souvikkai/souvik-ai-pm-portfolio/blob/main/day22-qlora-finetuning-lab/Day_22_QLoRA_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch, platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

In [ ]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate

In [ ]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    get_peft_model,
)

from trl import SFTTrainer
from datasets import Dataset

print("All imports successful.")

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

In [ ]:
raw_examples = [
    {
        "question": "Why is HBM bandwidth important for AI inference?",
        "answer": "HBM bandwidth is critical because GPUs must continuously move activations and KV cache data during inference. Insufficient bandwidth can starve compute units and increase latency."
    },
    {
        "question": "What is KV cache in transformers?",
        "answer": "KV cache stores attention keys and values from previous tokens so the model does not recompute them during autoregressive generation."
    },
    {
        "question": "Why does P99 latency matter in AI systems?",
        "answer": "P99 latency captures worst-case user experience. High tail latency can make AI applications feel unreliable even if average latency is low."
    },
    {
        "question": "Why use LoRA instead of full fine-tuning?",
        "answer": "LoRA reduces GPU memory and training cost by updating small low-rank adapter matrices instead of all model weights."
    },
]

formatted_examples = []

for ex in raw_examples:
    messages = [
        {
            "role": "system",
            "content": "You are a concise AI infrastructure and semiconductor product expert. Answer like a senior technical PM."
        },
        {
            "role": "user",
            "content": ex["question"]
        },
        {
            "role": "assistant",
            "content": ex["answer"]
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    formatted_examples.append({"text": text})

dataset = Dataset.from_list(formatted_examples)

print(dataset)
print(dataset[0]["text"])

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(bnb_config)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded.")
print("Device:", model.device)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen-day22-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

print(training_args)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Trainer created.")

In [ ]:
trainer.train()

In [ ]:
from collections import Counter

dtype_counts = Counter()

for name, param in model.named_parameters():
    if param.requires_grad:
        dtype_counts[str(param.dtype)] += param.numel()

print(dtype_counts)

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.dtype, param.shape)
        break

In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

from collections import Counter

dtype_counts = Counter()
for name, param in model.named_parameters():
    if param.requires_grad:
        dtype_counts[str(param.dtype)] += param.numel()

print(dtype_counts)

In [ ]:
trainer.train()

In [ ]:
prompt = "Why is KV cache important for LLM inference?"

messages = [
    {
        "role": "system",
        "content": "You are a concise AI infrastructure and semiconductor product expert. Answer like a senior technical PM."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=0.7,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

In [ ]:
prompt = "What is KV cache in transformers?"

messages = [
    {
        "role": "system",
        "content": "You are a concise AI infrastructure and semiconductor product expert. Answer like a senior technical PM."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=False,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)

total_memory = gpu_stats.total_memory / 1024**3
reserved_memory = torch.cuda.max_memory_reserved() / 1024**3
allocated_memory = torch.cuda.max_memory_allocated() / 1024**3

print(f"GPU: {gpu_stats.name}")
print(f"Total GPU memory: {total_memory:.2f} GB")
print(f"Max reserved memory: {reserved_memory:.2f} GB")
print(f"Max allocated memory: {allocated_memory:.2f} GB")

In [ ]:
model.save_pretrained("./qwen-day22-lora-adapter")
tokenizer.save_pretrained("./qwen-day22-lora-adapter")

print("Saved LoRA adapter and tokenizer.")

In [ ]:
!ls -lh ./qwen-day22-lora-adapter

In [ ]:
!zip -r qwen-day22-lora-adapter.zip qwen-day22-lora-adapter
!ls -lh qwen-day22-lora-adapter.zip